In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.feature_extraction.text import TfidfVectorizer
sys.path.append(os.path.abspath("../.."))
from ai.model.feature_extractor import extract_features


In [ ]:
df_clean = pd.read_csv("../data/processed/dataset.csv")
print(f"Dataset size: {len(df_clean)}")
print(df_clean.head())


In [ ]:
print("Extracting 104 mathematical features...")
features_104 = df_clean["integrand"].apply(extract_features)
features_104 = features_104.dropna()
df_104 = pd.DataFrame(features_104.tolist())
print(f"Mathematical features shape: {df_104.shape}")


In [ ]:
print("Fitting TF-IDF Vectorizer for 446 text-based features...")
vectorizer = TfidfVectorizer(max_features=446, analyzer='char_wb', ngram_range=(1, 4))
integrands = df_clean.loc[df_104.index, "integrand"]
X_tfidf = vectorizer.fit_transform(integrands).toarray()

feature_names = [f"tfidf_{i}" for i in range(446)]
df_tfidf = pd.DataFrame(X_tfidf, columns=feature_names, index=df_104.index)
print(f"TF-IDF features shape: {df_tfidf.shape}")


In [ ]:
df_550 = pd.concat([df_104, df_tfidf], axis=1)
df_550["label"] = df_clean.loc[df_104.index, "action"].fillna(0).astype(int)

print(f"Final 550 features dataset shape: {df_550.shape}")
print(f"Total features: {df_550.shape[1] - 1}")

output_path = "../data/processed/dataset_550.csv"
df_550.to_csv(output_path, index=False)
print(f"Saved 550 features dataset to: {output_path}")
